## Laboratorio N° 1 - Introducción a la Ciencia de Datos

### Montiel Nicolas, Lopez Gabriel.

# Punto 1 

### Lectura de datos
- Se crearon dos DataFrames que contienen los datos contenidos en los archivos .csv

- Como los archivos contienen datos con tildes primero se los tuvo codificar

- Una vez impresos ambos DataFrames se puede observar que las tablas contienen información sobre un seguimiento de ventas

- DataFrame1 con 1000 filas y DataFrame2 con 50

In [ ]:
import pandas as pd

dataFrame1 = pd.read_csv('datos_ventas_suc1.csv', encoding='latin-1')
dataFrame2 = pd.read_csv('datos_ventas_suc2.csv', encoding='latin-1')

print(dataFrame1)
print("________________________________________________________________")
print(dataFrame2)

### Fusión de Datos
- Se concatenó DataFrame2 a DataFrame1 juntando los datos de ambas tablas en una sola

- DataFrameFusionado tiene un total de 1050 filas demostrando que la fusión se realizó correctamente

In [ ]:
print("Df fusionado")
dataFrameFusionado = pd.concat([dataFrame1,dataFrame2], ignore_index=True)
print(dataFrameFusionado)

### Tratamiento de Datos
- Se transformó el formato de la columna 'Fecha' dd/MM/YYYY a YYYY-MM-dd

In [ ]:
#Tratamiento de datos
dataFrameFusionado['Fecha'] = pd.to_datetime(dataFrameFusionado['Fecha'])
print(dataFrameFusionado)

### Análisis de ventas
- Se hizo uso de la función groupBy() para realizar análisis de ventas

- Se averiguó el producto más vendido dando como resultado que la categoria Alimentos fue la que más cantidad de productos ha vendido

- Se averiguó el mes con más ventas dando como resultado al mes de Agosto

In [ ]:
#Producto más vendido
print("Tabla cantidad de productos totales")
productoMasVendido = dataFrameFusionado.groupby('Producto')['Cantidad'].sum()
print(productoMasVendido.sort_values(ascending=False))

In [ ]:
#Mes con más ventas
mesVentas = dataFrameFusionado.groupby(dataFrameFusionado['Fecha'].dt.month_name())['Total Venta'].sum()

print(mesVentas.sort_values(ascending=False))

### Datos agrupados
- Datos agrupados por categoría de producto

- Dió como resultado que la categoría de electrónica fue la que más ingresos generó, aún cuando vimos en los análisis anteriores que no fue la categoria que más cantidad de productos vendió.

- La categoria que menos ingresos generó fue la de Alimentos siendo la categoria que más cantidad de productos vendió

In [ ]:
#Ventas por categoria de producto
ventasPorCategorias = dataFrameFusionado.groupby('Producto')['Total Venta'].sum()

print(ventasPorCategorias.sort_values(ascending=False))

### Guardar Resultados

- Para guardar los DataFrame con la libreria pandas en formato .csv se hace uso de la funcion to_csv() junto dos parámetros: String Nombre archivo, Boolean index

- Como resultado se guarda el archivo en formato .csv con el nomrbe pasado por parámetro y sin la columna que marca el índice de las filas.

In [ ]:
dataFrameFusionado.to_csv('datos_ventas_fusionados.csv', index=False)

# Punto 2 - Matplotlib

### Gráfico de Torta
- Configuramos el plot para mostrar un gráfico de torta con el motivo de expresar en porcentajes la cantidad de productos vendidos por categoría


In [ ]:
import matplotlib.pyplot as plt
##Gráfico de torta
ventasPorCategorias = dataFrameFusionado.groupby('Producto')['Total Venta'].sum().reset_index()

plt.figure(figsize=(8,8))

plt.pie(ventasPorCategorias['Total Venta'], labels=ventasPorCategorias['Producto'], autopct='%1.1f%%') #Formato de porcentaje con un solo decimal
plt.title('Ventas por categoría')
plt.axis('equal')

### Gráfico de Lineas
- Con el motivo de mostrar el eje x con los meses ordenados, se hizo lo siguiente:

In [ ]:
#Ordenando meses
ordenMeses = ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
ventasPorMes = dataFrameFusionado.groupby(dataFrameFusionado['Fecha'].dt.month_name())['Total Venta'].sum().reset_index()
ventasPorMes['Fecha'] = pd.Categorical(ventasPorMes['Fecha'], categories=ordenMeses, ordered=True)
ventasPorMes = ventasPorMes.sort_values('Fecha')
print(ventasPorMes)

Una vez ordenado los meses fuimos capaces de configurar el plot.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))
plt.plot(ventasPorMes['Fecha'], ventasPorMes['Total Venta'], marker='o')
plt.title('Ventas por mes')
plt.xlabel('Mes')
plt.ylabel('Total Venta')
plt.xticks(rotation=45)
plt.grid()
plt.show()  

# Punto 3 - Dataset Ajedrez

Para el siguiente punto se eligió un dataset de la página **kaggle.com** que consiste en 20000 datos sobre partidas de ajedrez.

Cada partida se compone de los siguientes datos:
- Id de la partida
- Si los jugadores están certificados
- Hora de inicio
- Hora de fin
- Número de turnos
- Tipo de victoria (Tiempo, Jaque Mate, Por oponente rendido)
- Ganador
- Tiempo de partida
- Id de Jugador Blancas
- Elo de Jugador Blancas
- Id de Jugador Negras
- Id de Jugador Negras
- Todos los movimientos de la partida en formato SCN (Standar Chess Notation)
- Nombre de la apertura
- Jugadas de apertura (Numero de movimientos en fase de apertura)

In [ ]:
dataFrameAjedrez = pd.read_csv('games.csv')

print(dataFrameAjedrez)

## ¿Qué jugadores se rinden con más facilidad?

Teniendo a disposición el número de turnos que duró la partida y el tipo de victoria decidimos averiguar que tipo de jugadores son los que se rinden con más facilidad, teniendo dos hipótesis:

1) Los jugadores con menos elo cometen más errores y se frustran con más facilidad, lo que los hace rendirse a la mínima situación en las que se ven comprometidos / Los jugadores con más elo no cometen tantos errores que comprometan la partida tan rápido, por lo que posterga la decisión de rendirse en instancias finales de la partida.

2) Los jugadores con más elo son capaces de ver más movimientos en el futuro por lo que pueden reconocer que serán derrotados y no pierden más el tiempo tomando la decisión de rendirse / Los jugadores con menos elo no saben que van a perder por lo que siguen jugando y la partida termina en jaque mate en vez de que se rindan.


In [ ]:
import numpy as np

# Preparación del DataFrame

# Filtramos partidas que terminaron en 
partidasRendidas = dataFrameAjedrez[dataFrameAjedrez['victory_status'] == 'resign'].copy()

# Separamos el elo del jugador perdedor
partidasRendidas['elo_perdedor'] = np.where(partidasRendidas['winner'] == 'white', partidasRendidas['black_rating'], partidasRendidas['white_rating'])

# Agrupamos el elo en rangos para que luego podamos ver una tendencia
bins = [0, 1000, 1250, 1500, 1750, 2000, 2250, 2500, 3000]
labels = ['<1000', '1000-1250', '1250-1500', '1500-1750', '1750-2000', '2000-2250', '2250-2500', '>2500']
partidasRendidas['elo_bin'] = pd.cut(partidasRendidas['elo_perdedor'], bins=bins, labels=labels)

# Calculamos el promedio de turnos que tardó cada grupo antes de rendirse
turnos_por_elo = partidasRendidas.groupby('elo_bin', observed=False)['turns'].mean().reset_index()

print(turnos_por_elo)

In [ ]:
# El gráfico queda de la siguiente manera
plt.figure(figsize=(10,6))
plt.plot(turnos_por_elo['elo_bin'], turnos_por_elo['turns'], marker='o', color='blue', linewidth=2)
plt.title("Promedio de turnos antes de rendirse por nivel del jugador")
plt.xlabel("Elo del Jugador que se rinde")
plt.ylabel("Promedio de Turnos para rendirse")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


Se puede observar que en cuanto más aumenta el elo del perdedor se promedian más turnos en la partida, por lo que se puede concluir que la primera hipótesis es la mas acertada.